# Live acceptance through voltsource.AD5780
This notebook is the acceptance record for the live-validation goal. Every DAC operation uses the class exported from `malab.instruments.voltsource`. Tests and assertions are in the cells, not hidden case dispatchers. Supporting diagnostic cases are labeled and cannot substitute for final acceptance. All four outputs must be isolated; only CH1 is connected to the meter. Keep the Â±0.5 mV voltage criterion.

The command-line runner executes these same cells and saves an executed notebook plus a manifest. Default settings prevent hardware access. Measurements and failures are retained in `notebooks/results`. Do not run two instrument notebooks simultaneously.


In [ ]:
from pathlib import Path
import sys, time, math, csv, json
from datetime import datetime, timezone
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"malab/instruments/ad5780.py").exists())
sys.path.insert(0, str(root))
import malab
assert Path(malab.__file__).resolve().parent == root/"malab", "Restart kernel: another malab is imported"
from malab.instruments.voltsource import AD5780
from malab.instruments import Keysight34461A
from malab.instruments.ad5780 import V_PER_BIT, MAX_CODE, voltage_to_code, code_to_voltage
from malab.instruments._line_socket import InstrumentError, CommandError, ProtocolError, InstrumentTimeout
print("Acceptance API:", AD5780, "via malab.instruments.voltsource")
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RESULTS = root/"malab/notebooks/results"/RUN_ID
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
RUN_LIVE = False
ALL_OUTPUTS_ISOLATED = False
from malab.bench_config import load_bench_config
BENCH = load_bench_config() if RUN_LIVE else None
DAC_ADDRESS = BENCH["units"][0]["address"] if BENCH else None
METER_ADDRESS = BENCH["meter"]["address"] if BENCH else None
USB_PORT = BENCH["units"][0].get("usb_port") if BENCH else None
CHANNEL = 1
VOLTAGE_TOL = 0.0005
RATE_REL_TOL = 0.05
DAC_OPTIONS = dict(address=DAC_ADDRESS, timeout=3)


In [ ]:
def ready():
    assert RUN_LIVE and ALL_OUTPUTS_ISOLATED, "Live work is disabled"
    assert CHANNEL == 1
def save_rows(name, rows):
    path = RESULTS/(name+".csv")
    if rows:
        with path.open("w",newline="") as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]))
            writer.writeheader(); writer.writerows(rows)
    print("Saved",len(rows),"measurements:",path)
def connect_dac(timeout=20, **extra):
    deadline=time.monotonic()+timeout
    last=None
    while time.monotonic()<deadline:
        try:
            return AD5780(**(DAC_OPTIONS|extra))
        except (InstrumentError,OSError) as exc:
            last=exc
            time.sleep(0.2)
    raise TimeoutError(f"Fresh connection failed: {last}")



## Diagnostic D01 â€” uncorrected voltage survey (not final acceptance)
Measure raw nominal endpoints and intermediate levels in both directions. Repeat zero to expose drift/hysteresis. A least-squares line is a candidate calibration; independent validation points must pass afterward. This case reports the current Â±0.5 mV failures instead of changing the tolerance.


In [ ]:
ready()
rows=[]
try:
    with connect_dac(use_calibration=False) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=10)
        print(dac.initialize(CHANNEL))
        print("Meter:",meter.get_id())
        for requested in (0,-10,-8,-4,0,4,8,10,0,8,-8,0):
            dac.set_voltage(CHANNEL,requested)
            time.sleep(0.3)
            actual_code=dac.get_code(CHANNEL)
            reported=dac.get_voltage(CHANNEL)
            values=[meter.read_voltage() for _ in range(3)]
            measured=sum(values)/len(values)
            row=dict(requested_V=requested,code=actual_code,reported_V=reported,
                     measured_V=measured,error_V=measured-reported,spread_V=max(values)-min(values))
            rows.append(row); print(row,flush=True)
        dac.set_voltage(CHANNEL,0)
        meter.check_errors()
finally:
    save_rows("raw_voltage_survey",rows)
xs=[r["code"] for r in rows]; ys=[r["measured_V"] for r in rows]
xm=sum(xs)/len(xs); ym=sum(ys)/len(ys)
gain=sum((x-xm)*(y-ym) for x,y in zip(xs,ys))/sum((x-xm)**2 for x in xs)
intercept=ym-gain*xm
residuals=[y-(gain*x+intercept) for x,y in zip(xs,ys)]
candidate=dict(volts_per_code=gain,volts_at_code_zero=intercept,
               zero_code=round(-intercept/gain),min_voltage=intercept,
               max_voltage=intercept+MAX_CODE*gain,
               max_residual=max(map(abs,residuals)),
               nominal_zero_offset=intercept+131072*gain)
(RESULTS/"calibration_candidate.json").write_text(json.dumps(candidate,indent=2)+"\n")
print("Candidate calibration:",candidate)
assert all(abs(r["error_V"])<=VOLTAGE_TOL for r in rows), "Uncorrected voltage survey fails Â±0.5 mV; calibration/diagnosis required"



## Diagnostic D02 â€” ramp timing before repair
The public ramp method starts a 0.2 V sweep at 0.1 V/s. Meter change is evaluated relative to its own initial reading so an offset does not disguise the timing defect. Final absolute accuracy remains a separate required acceptance.


In [ ]:
ready()
rows=[]
try:
    with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize(CHANNEL)
        v0=dac.set_voltage(CHANNEL,0)
        time.sleep(0.2)
        measured0=meter.read_voltage()
        start_code=dac.get_code(CHANNEL)
        target=dac.voltage_to_code(CHANNEL,0.2)
        expected=math.ceil(abs(target-start_code)/10)*math.ceil(10*dac.get_calibration(CHANNEL).volts_per_code/0.1*1e6)/1e6
        t0=time.monotonic()
        accepted=dac.ramp(CHANNEL,0.2,0.1,wait=False)
        while True:
            ta=time.monotonic(); measured=meter.read_voltage(); tb=time.monotonic()
            status=dac.get_status()
            row=dict(elapsed_s=(ta+tb)/2-t0,measured_V=measured,delta_V=measured-measured0,state=status.state)
            rows.append(row)
            assert status.state in ("RUNNING","COMPLETE"), status
            if status.state=="COMPLETE":
                duration=time.monotonic()-t0
                break
            assert time.monotonic()-t0<10,"Ramp did not finish"
        final_code=dac.get_code(CHANNEL)
        dac.set_voltage(CHANNEL,0)
        print(dict(expected_s=expected,actual_s=duration,ratio=duration/expected,final_code=final_code),flush=True)
        assert final_code==target
        assert expected-0.05<=duration<=expected*(1+RATE_REL_TOL)+0.1, "Main API ramp timing fails"
finally:
    save_rows("ramp_timing_baseline",rows)



## V01 — final calibrated SET/GET accuracy
Independent validation points, repeated zero, exact register endpoint and explicit unattainable-voltage rejection. ±0.5 mV stays unchanged.


In [ ]:
ready()
rows=[]
try:
    with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=10)
        dac.initialize(CHANNEL)
        profile=dac.get_calibration(CHANNEL)
        print("Device calibration:",profile)
        assert profile.measured, "CH1 calibration was not loaded from firmware"
        print("Achievable requested range:",dac.get_voltage_limits(CHANNEL))
        for requested in (0, -9, -3.3, -0.05, 0, 0.05, 0.123, 3.3, 9, 10, 0):
            expected=dac.set_voltage(CHANNEL,requested)
            time.sleep(0.2)
            code=dac.get_code(CHANNEL)
            reported=dac.get_voltage(CHANNEL)
            values=[meter.read_voltage() for _ in range(3)]
            measured=sum(values)/len(values)
            row=dict(requested_V=requested,code=code,reported_V=reported,measured_V=measured,
                     error_to_request_V=measured-requested,error_to_get_V=measured-reported)
            rows.append(row); print(row,flush=True)
            assert code==dac.voltage_to_code(CHANNEL,requested), row
            assert reported==expected, row
            assert all(abs(v-requested)<=VOLTAGE_TOL and abs(v-reported)<=VOLTAGE_TOL for v in values), row
        before=dac.get_code(CHANNEL)
        try:
            dac.set_voltage(CHANNEL,-10)
        except ValueError as exc:
            print("Correctly rejected unattainable endpoint:",exc)
        else:
            raise AssertionError("Calibrated -10 V cannot be generated by this hardware")
        assert dac.get_code(CHANNEL)==before
        meter.check_errors()
finally:
    save_rows("calibrated_voltage_acceptance",rows)
print("PASS V01: calibrated SET and GET agree with independent meter within 0.5 mV")


## R01 — Final ramp rates, paths, endpoints and synchronous completion


In [ ]:
ready()
rows=[]; summaries=[]
try:
    with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize(CHANNEL)
        cal=dac.get_calibration(CHANNEL)
        scenarios=[(speed,span,10) for speed,span in ((0.001,0.02),(0.01,0.05),(0.1,0.2))]
        scenarios += [(0.1,0.2,step) for step in (1,100)]
        for speed,span,step in scenarios:
            for direction in (-1,1):
                target_v=direction*span
                v0=dac.set_voltage(CHANNEL,0)
                time.sleep(0.1)
                initial=dac.get_code(CHANNEL)
                target=dac.voltage_to_code(CHANNEL,target_v)
                delay=math.ceil(step*cal.volts_per_code/speed*1e6)
                expected=math.ceil(abs(target-initial)/step)*delay/1e6
                run=f"speed={speed:g},step={step},target={target_v:g}"
                points=[]
                t0=time.monotonic()
                accepted=dac.ramp(CHANNEL,target_v,speed,step_size=step,wait=False)
                ack_s=time.monotonic()-t0
                assert accepted.state=="ACCEPTED" and ack_s<0.5,(accepted,ack_s)
                while True:
                    ta=time.monotonic(); measured=meter.read_voltage(); tb=time.monotonic()
                    status=dac.get_status()
                    row=dict(run=run,elapsed_s=(ta+tb)/2-t0,measurement_s=tb-ta,measured_V=measured,state=status.state)
                    rows.append(row); points.append(row)
                    assert (status.channel,status.target_code)==(CHANNEL,target),status
                    assert status.state in ("RUNNING","COMPLETE"),status
                    if status.state=="COMPLETE":
                        elapsed=time.monotonic()-t0
                        break
                    assert time.monotonic()-t0<expected*1.5+3,"Ramp overdue"
                final=dac.get_voltage(CHANNEL)
                final_meter=meter.read_voltage()
                gap=max([p["measurement_s"] for p in points]+[b["elapsed_s"]-a["elapsed_s"] for a,b in zip(points,points[1:])])
                summary=dict(run=run,expected_s=expected,elapsed_s=elapsed,ratio=elapsed/expected,ack_s=ack_s,final_V=final,measured_V=final_meter)
                summaries.append(summary); print(summary,flush=True)
                assert expected-0.05<=elapsed<=expected*(1+RATE_REL_TOL)+2*gap+0.05,summary
                assert dac.get_code(CHANNEL)==target
                assert abs(final_meter-target_v)<=VOLTAGE_TOL and abs(final_meter-final)<=VOLTAGE_TOL,summary
                assert len(points)>=5
                for point in points:
                    ideal=v0+direction*min(abs(final-v0),speed*point["elapsed_s"])
                    tolerance=VOLTAGE_TOL+step*cal.volts_per_code+speed*point["measurement_s"]+RATE_REL_TOL*abs(final-v0)
                    assert abs(point["measured_V"]-ideal)<=tolerance,(point,ideal,tolerance)
                    assert min(v0,final)-VOLTAGE_TOL<=point["measured_V"]<=max(v0,final)+VOLTAGE_TOL,point
                for a,b in zip(points,points[1:]):
                    assert direction*(b["measured_V"]-a["measured_V"])>=-2*VOLTAGE_TOL,(a,b)
        # Default wait=True must return completion, then the next read must be a READ reply.
        finished=dac.ramp(CHANNEL,0,0.1)
        assert finished.state=="COMPLETE"
        assert abs(dac.get_voltage(CHANNEL))<VOLTAGE_TOL
        assert abs(meter.read_voltage())<VOLTAGE_TOL
        meter.check_errors()
finally:
    save_rows("ramp_acceptance_paths",rows)
    save_rows("ramp_acceptance_timing",summaries)
print("PASS R01: all rates, step sizes, directions, paths and endpoints")


## N01 — NETWORK through the main API preserves outputs and interrupts a ramp


In [ ]:
ready()
rows=[]
try:
    with connect_dac(bench_port=USB_PORT) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize()
        held=dac.set_voltage(CHANNEL,0.03)
        time.sleep(0.1)
        before=[dac.get_code(ch) for ch in range(1,5)]
        assert abs(meter.read_voltage()-held)<VOLTAGE_TOL
        print(dac.network_reset(wait=True),flush=True)
        assert dac.get_id()=="AD5780"
        assert [dac.get_code(ch) for ch in range(1,5)]==before
        for _ in range(8):
            measured=meter.read_voltage()
            rows.append(dict(phase="hold_after_reset",measured_V=measured,expected_V=held))
            assert abs(measured-held)<VOLTAGE_TOL
        dac.set_voltage(CHANNEL,0)
        dac.ramp(CHANNEL,0.05,0.01,wait=False)
        deadline=time.monotonic()+3
        while meter.read_voltage()<0.01:
            assert time.monotonic()<deadline,"Ramp did not progress"
        print(dac.network_reset(wait=True),flush=True)
        assert dac.get_status().state=="ABORTED"
        held=dac.get_voltage(CHANNEL)
        assert 0.005<held<0.045,held
        for _ in range(12):
            measured=meter.read_voltage()
            rows.append(dict(phase="aborted_ramp",measured_V=measured,expected_V=held))
            assert abs(measured-held)<VOLTAGE_TOL
            time.sleep(0.05)
        assert [dac.get_code(ch) for ch in (2,3,4)]==before[1:]
        dac.set_voltage(CHANNEL,0)
finally:
    save_rows("network_acceptance",rows)
print("PASS N01: output preservation, fresh connection, and ramp interruption via main API")


## S01 — SAFE ZERO from both polarities without Ethernet, priority and latch


In [ ]:
ready()
rows=[]; summaries=[]
try:
    with connect_dac(bench_port=USB_PORT) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize()
        zero_codes=[dac.get_calibration(ch).zero_code for ch in range(1,5)]
        for start_v in (0.05,-0.05):
            # Other channels are isolated; arrange small opposite-sign register offsets too.
            for ch in (2,3,4):
                dac.set_voltage(ch, (1 if ch%2 else -1)*0.01)
            v0=dac.set_voltage(CHANNEL,start_v)
            time.sleep(0.1)
            assert abs(meter.read_voltage()-v0)<VOLTAGE_TOL
            dac.close()  # Explicitly end Ethernet before requesting the local action.
            t0=time.monotonic()
            print(dac.safe_zero(),flush=True)
            assert not dac.connected
            try:
                dac.network_reset()
            except CommandError:
                pass
            else:
                raise AssertionError("NETWORK overrode active SAFE ZERO")
            points=[]
            while True:
                ta=time.monotonic(); measured=meter.read_voltage(); tb=time.monotonic()
                state=dac.get_recovery_status()
                row=dict(start_V=v0,elapsed_s=(ta+tb)/2-t0,measurement_s=tb-ta,measured_V=measured,state=state)
                rows.append(row); points.append(row)
                assert state in ("ACTIVE","LATCHED"),state
                if state=="LATCHED":
                    elapsed=time.monotonic()-t0
                    break
                assert time.monotonic()-t0<10,"SAFE ZERO overdue"
            expected=abs(v0)/0.02
            final=meter.read_voltage()
            summary=dict(start_V=v0,expected_s=expected,elapsed_s=elapsed,final_V=final)
            summaries.append(summary); print(summary,flush=True)
            gap=max([p["measurement_s"] for p in points]+[b["elapsed_s"]-a["elapsed_s"] for a,b in zip(points,points[1:])])
            assert expected-0.08<=elapsed<=expected*(1+RATE_REL_TOL)+2*gap+0.1,summary
            assert abs(final)<=VOLTAGE_TOL,summary
            direction=1 if v0>0 else -1
            for point in points:
                ideal=direction*max(0,abs(v0)-0.02*point["elapsed_s"])
                tolerance=VOLTAGE_TOL+0.02*point["measurement_s"]+RATE_REL_TOL*abs(v0)
                assert abs(point["measured_V"]-ideal)<=tolerance,(point,ideal,tolerance)
                assert min(v0,0)-VOLTAGE_TOL<=point["measured_V"]<=max(v0,0)+VOLTAGE_TOL,point
            for a,b in zip(points,points[1:]):
                assert direction*(b["measured_V"]-a["measured_V"])<=2*VOLTAGE_TOL,(a,b)
            time.sleep(0.5)
            assert dac.get_recovery_status()=="LATCHED"
            try:
                with AD5780(address=DAC_ADDRESS,timeout=0.5) as blocked:
                    blocked.get_id()
            except InstrumentError:
                pass
            else:
                raise AssertionError("Latched controller accepted an Ethernet client")
            print(dac.network_reset(wait=True),flush=True)
            assert [dac.get_code(ch) for ch in range(1,5)]==zero_codes
            assert abs(meter.read_voltage())<=VOLTAGE_TOL
        # Blocking completion must also be available on the main class.
        dac.set_voltage(CHANNEL,0.005)
        assert dac.safe_zero(wait=True,timeout=3)=="LATCHED"
        assert abs(meter.read_voltage())<=VOLTAGE_TOL
        dac.network_reset(wait=True)
        assert [dac.get_code(ch) for ch in range(1,5)]==zero_codes
finally:
    save_rows("safe_zero_acceptance_path",rows)
    save_rows("safe_zero_acceptance_timing",summaries)
print("PASS S01: network-independent safe-zero, both signs, latch, priority and rearm")


## F01 — Invalid requests and oversized input preserve outputs; clamping is reported


In [ ]:
ready()
with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    dac.initialize()
    held=dac.set_voltage(CHANNEL,0.02)
    before=[dac.get_code(ch) for ch in range(1,5)]
    for channel in (0,5,-1,True,1.5):
        try:
            dac.set_voltage(channel,0)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid channel accepted: {channel}")
    for voltage in (-11,11,float("nan"),float("inf"),None):
        try:
            dac.set_voltage(CHANNEL,voltage)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid voltage accepted: {voltage}")
    for rate in (0,-1,float("nan"),float("inf")):
        try:
            dac.ramp(CHANNEL,0,rate)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid rate accepted: {rate}")
    rejected=("READ 0","READ 5","SET 0 0","SET 5 0","RAMP 0 0 1 1",
        "RAMP 1 0 -1 1","RAMP 1 0 0 1","RAMP 1 0 262144 1","RAMP 1 0 1 -1",
        "READ +","READ -","SET 1 12x","SET 1 2147483648","SET 1 -2147483649",
        "RAMP 1 0 1 2147483648","READ 4294967297")
    for command in rejected:
        try:
            dac.query(command)
        except CommandError as exc:
            print(command,"->",exc)
        else:
            raise AssertionError(f"Firmware accepted {command}")
        assert dac.get_id()=="AD5780"
        assert [dac.get_code(ch) for ch in range(1,5)]==before
    # Main API's explicit low-level write reaches the device input-length guard.
    dac.write("SET 1 0"+" "*150+"trailing")
    rejected_reply=dac.read()
    assert rejected_reply.startswith("ERR!"),rejected_reply
    assert dac.get_id()=="AD5780"
    assert [dac.get_code(ch) for ch in range(1,5)]==before
    assert abs(meter.read_voltage()-held)<VOLTAGE_TOL
    for requested,clamped in ((262144,262143),(-1,0)):
        reply=dac.query(f"SET 1 {requested}")
        print(reply)
        assert reply==f"WARN! Target clamped to DAC range. SET command received with parameters: 1 {clamped}",reply
        assert dac.get_code(CHANNEL)==clamped
        assert abs(meter.read_voltage()-dac.get_voltage(CHANNEL))<VOLTAGE_TOL
    reply=dac.query("RAMP 1 262144 262143 1000")
    assert reply.startswith("WARN! Target clamped to DAC range. RAMP ACCEPTED 1 262143"),reply
    deadline=time.monotonic()+2
    while dac.get_status().state=="RUNNING":
        assert time.monotonic()<deadline
    assert dac.get_code(CHANNEL)==262143
    dac.set_voltage(CHANNEL,0)
    assert abs(meter.read_voltage())<VOLTAGE_TOL
print("PASS F01: client/device validation, unchanged outputs on rejection, input recovery and warnings")


## C01 — Single-client ownership, connection churn and partial-input cleanup


In [ ]:
ready()
with connect_dac() as owner:
    owner.initialize(CHANNEL)
    held=owner.set_voltage(CHANNEL,0.02)
    initial=owner.get_code(CHANNEL)
    for attempt in range(100):
        try:
            with AD5780(address=DAC_ADDRESS,timeout=1) as unwanted:
                raise AssertionError("Second client accepted")
        except CommandError as exc:
            assert "busy" in str(exc).lower(),exc
        assert owner.get_id()=="AD5780"
        if attempt%20==0: print("Rejected extra clients:",attempt+1,flush=True)
    assert owner.get_code(CHANNEL)==initial
for _ in range(20):
    # Constructor receives the banner; no command is sent by this silent client.
    with connect_dac():
        pass
with connect_dac() as partial:
    partial.write("SET 1 ",terminate=False)
with connect_dac() as fresh:
    assert fresh.get_id()=="AD5780", "Partial command survived disconnect"
    assert fresh.get_code(CHANNEL)==initial
    fresh.write("READ ",terminate=False)
    fresh.write("1")
    assert int(fresh.read())==initial
    fresh.set_voltage(CHANNEL,0)
print("PASS C01: ownership survives churn; disconnected fragments are discarded")


## C02 — Busy lock, servicing, ownership during ramp, abort and no post-disconnect step


In [ ]:
ready()
with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=0.2)
    dac.initialize()
    dac.set_voltage(CHANNEL,0)
    accepted=dac.ramp(CHANNEL,0.05,0.01,wait=False)
    for operation in (lambda:dac.set_voltage(CHANNEL,0),lambda:dac.initialize(CHANNEL),
                      lambda:dac.get_voltage(CHANNEL),lambda:dac.ramp(CHANNEL,0,0.1)):
        try:
            operation()
        except CommandError as exc:
            assert "busy" in str(exc).lower(),exc
        else:
            raise AssertionError("Busy DAC operation accepted")
    for _ in range(5):
        try:
            with AD5780(address=DAC_ADDRESS,timeout=1):
                raise AssertionError("Second owner accepted during ramp")
        except CommandError:
            pass
        assert dac.get_id()=="AD5780"
        assert dac.get_status().state=="RUNNING"
    time.sleep(0.3)
    dac.abort_ramp()
with connect_dac() as fresh, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=0.2)
    assert fresh.get_status().state=="ABORTED"
    held=fresh.get_voltage(CHANNEL)
    assert 0<=held<0.05,held
    for _ in range(5):
        assert abs(meter.read_voltage()-held)<VOLTAGE_TOL
        time.sleep(0.1)
    fresh.set_voltage(CHANNEL,0)
    initial=fresh.get_code(CHANNEL)
    target_v=fresh.code_to_voltage(CHANNEL,initial+1)
    speed=fresh.get_calibration(CHANNEL).volts_per_code/20
    t0=time.monotonic()
    accepted=fresh.ramp(CHANNEL,target_v,speed,step_size=1,wait=False)
    assert time.monotonic()-t0<0.5,"Acceptance blocked behind long delay"
    time.sleep(0.2)
    fresh.abort_ramp()
with connect_dac() as final:
    assert final.get_status().state=="ABORTED"
    assert final.get_code(CHANNEL)==initial,"Extra step occurred after disconnect during delay"
print("PASS C02: busy lock, servicing, one owner, prompt ACK and disconnect cancellation")


## I01 — INIT preserves active outputs, channel isolation and repeated initialization


In [ ]:
ready()
rows=[]
with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    dac.initialize()
    for ch,v in enumerate((0.05,-0.02,0.01,-0.01),1):
        dac.set_voltage(ch,v)
    before=[dac.get_code(ch) for ch in range(1,5)]
    held=dac.get_voltage(CHANNEL)
    for channel in (1,2,3,4,None,None,1):
        reply=dac.initialize(channel)
        codes=[dac.get_code(ch) for ch in range(1,5)]
        measured=meter.read_voltage()
        rows.append(dict(initialized=str(channel),codes=str(codes),measured_V=measured))
        assert codes==before,(channel,codes,before)
        assert abs(measured-held)<VOLTAGE_TOL,(channel,measured,held)
        assert 'active outputs preserved' in reply
    # A channel-1 write/ramp must leave the other channels' registers unchanged.
    dac.ramp(CHANNEL,-0.05,0.1)
    assert [dac.get_code(ch) for ch in range(2,5)]==before[1:]
    assert abs(meter.read_voltage()+0.05)<VOLTAGE_TOL
    for ch in range(1,5): dac.set_voltage(ch,0)
    assert abs(meter.read_voltage())<VOLTAGE_TOL
save_rows('init_preservation',rows)
print('PASS I01: active INIT is read-only, repeatable, all-channel safe; channel isolation')

## L01 — 120-second idle lease expires; a >120-second ramp retains ownership


In [ ]:
ready()
rows=[]
with connect_dac() as idle, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    idle.initialize()
    held=idle.set_voltage(CHANNEL,0.02)
    code=idle.get_code(CHANNEL)
    t0=time.monotonic()
    # No DAC traffic during this wait. DMM traffic does not renew the DAC lease.
    while time.monotonic()-t0<123:
        measured=meter.read_voltage()
        rows.append(dict(phase='idle',elapsed_s=time.monotonic()-t0,measured_V=measured))
        assert abs(measured-held)<VOLTAGE_TOL
        print('Idle lease test:',round(time.monotonic()-t0,1),'s',flush=True)
        time.sleep(min(30,max(0,123-(time.monotonic()-t0))))
    try:
        idle.get_id()
    except InstrumentError as exc:
        print('Expired connection rejected:',type(exc).__name__,str(exc))
        assert not idle.connected
    else:
        raise AssertionError('Idle owner survived the 120-second lease')
with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    assert dac.get_code(CHANNEL)==code
    dac.set_voltage(CHANNEL,0)
    initial=dac.get_code(CHANNEL)
    cal=dac.get_calibration(CHANNEL)
    target_v=dac.code_to_voltage(CHANNEL,initial+1)
    t0=time.monotonic()
    accepted=dac.ramp(CHANNEL,target_v,cal.volts_per_code/125,step_size=1,wait=False)
    assert accepted.state=='ACCEPTED' and time.monotonic()-t0<0.5
    # No status polling/heartbeats for 127 s: the active operation must renew its lease.
    while time.monotonic()-t0<127:
        measured=meter.read_voltage()
        rows.append(dict(phase='active',elapsed_s=time.monotonic()-t0,measured_V=measured))
        assert abs(measured)<VOLTAGE_TOL
        print('Long-ramp lease test:',round(time.monotonic()-t0,1),'s',flush=True)
        time.sleep(min(30,max(0,127-(time.monotonic()-t0))))
    status=dac.get_status()
    assert status.state=='COMPLETE',status
    assert dac.get_id()=='AD5780'
    assert dac.get_code(CHANNEL)==initial+1
    dac.set_voltage(CHANNEL,0)
    assert abs(meter.read_voltage())<VOLTAGE_TOL
save_rows('idle_and_active_lease',rows)
print('PASS L01: idle ownership expires, long operation keeps ownership without heartbeats')

## Q01 — 6,000-command soak with USB monitor closed and unchanged output


In [ ]:
ready()
rows=[]
with connect_dac() as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    dac.initialize()
    held=dac.set_voltage(CHANNEL,0.02)
    codes=[dac.get_code(ch) for ch in range(1,5)]
    started=time.monotonic()
    for i in range(2000):
        assert dac.get_id()=='AD5780'
        assert dac.get_code(CHANNEL)==codes[CHANNEL-1]
        assert dac.get_status().state!='RUNNING'
        if (i+1)%100==0:
            measured=meter.read_voltage()
            assert abs(measured-held)<VOLTAGE_TOL
            rows.append(dict(commands=3*(i+1),elapsed_s=time.monotonic()-started,measured_V=measured))
            print('Soak:',3*(i+1),'commands',round(time.monotonic()-started,2),'s',flush=True)
    assert [dac.get_code(ch) for ch in range(1,5)]==codes
with connect_dac() as fresh:
    assert fresh.get_id()=='AD5780'
    assert [fresh.get_code(ch) for ch in range(1,5)]==codes
    fresh.set_voltage(CHANNEL,0)
save_rows('command_soak',rows)
print('PASS Q01: 6,000 commands, subsequent fresh connection, outputs held; heap audit is separate')

## E01 — Readback faults reject READ, INIT, RAMP and SAFE ZERO without changing outputs


In [ ]:
ready()
with connect_dac(bench_port=USB_PORT) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    meter.configure_dc_voltage(10,nplc=1)
    dac.initialize()
    held=dac.set_voltage(CHANNEL,0.02)
    before=[dac.get_code(ch) for ch in range(1,5)]
    baseline=meter.read_voltage()
    print('Pre-fault meter:',baseline,'expected:',held)
    assert abs(baseline-held)<VOLTAGE_TOL,(baseline,held)
    try:
        dac.bench_inject_fault('readback')
        for label,operation in [('READ',lambda:dac.get_voltage(CHANNEL)),
                                ('INIT',lambda:dac.initialize(CHANNEL)),
                                ('RAMP',lambda:dac.ramp(CHANNEL,0,0.1))]:
            try:
                operation()
            except CommandError as exc:
                print(label,'rejected:',exc)
            else:
                raise AssertionError(label+' accepted invalid readback')
            measured=meter.read_voltage()
            print(label,'held meter:',measured)
            assert abs(measured-held)<VOLTAGE_TOL,(label,measured,held)
        try:
            dac.safe_zero(wait=True,timeout=3)
        except CommandError as exc:
            assert 'failed validation' in str(exc),exc
        else:
            raise AssertionError('Invalid starting readback was reported as successful SAFE ZERO')
        assert dac.get_recovery_status()=='FAILED'
        assert abs(meter.read_voltage()-held)<VOLTAGE_TOL
    finally:
        dac.bench_inject_fault('clear')
        dac.network_reset(wait=True)
    assert [dac.get_code(ch) for ch in range(1,5)]==before
    assert abs(meter.read_voltage()-held)<VOLTAGE_TOL
    dac.set_voltage(CHANNEL,0)
    assert abs(meter.read_voltage())<VOLTAGE_TOL
print('PASS E01: simulated corrupt SPI replies fail explicitly; actual DAC outputs remain unchanged')

## E02 — Automatic recovery from real W5500 IP-register corruption preserves DACs


In [ ]:
ready()
rows=[]
for during_ramp in (False,True):
    with connect_dac(bench_port=USB_PORT) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=0.2)
        dac.initialize()
        dac.set_voltage(CHANNEL,0.02)
        before=[dac.get_code(ch) for ch in range(1,5)]
        assert abs(meter.read_voltage()-0.02)<VOLTAGE_TOL
        if during_ramp:
            dac.ramp(CHANNEL,0.05,0.01,wait=False)
            time.sleep(0.3)
        if not during_ramp: dac.write('SET 1 ',terminate=False)
        dac.bench_inject_fault('network')
        # Do not request NETWORK: the normal firmware health/retry path must recover.
        time.sleep(0.5)
        try:
            dac.get_id()
        except InstrumentError as exc:
            print('Old connection invalidated:',type(exc).__name__)
        else:
            raise AssertionError('Network corruption retained the old owner')
    with connect_dac() as fresh, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10,nplc=1)
        assert fresh.get_id()=='AD5780'
        after=[fresh.get_code(ch) for ch in range(1,5)]
        held=fresh.get_voltage(CHANNEL)
        assert after[1:]==before[1:]
        if during_ramp:
            assert fresh.get_status().state=='ABORTED'
            assert before[0]<=after[0]<fresh.voltage_to_code(CHANNEL,0.05)
        else:
            assert after==before
        for _ in range(5):
            measured=meter.read_voltage()
            rows.append(dict(during_ramp=during_ramp,held_code=after[0],held_V=held,measured_V=measured))
            assert abs(measured-held)<VOLTAGE_TOL,(during_ramp,measured,held)
        fresh.set_voltage(CHANNEL,0)
save_rows('automatic_network_recovery',rows)
print('PASS E02: actual W5500 configuration loss recovers automatically, aborts remote work, holds outputs')

## P01 — Public API completion/transport errors and no command replay


In [ ]:
ready()
with connect_dac() as dac:
    dac.initialize()
    dac.set_voltage(CHANNEL,0)
    try:
        dac.ramp(CHANNEL,0.05,0.01,timeout=0.05)
    except InstrumentTimeout as exc:
        assert 'completion deadline' in str(exc),exc
        assert dac.connected and dac.get_status().state=='RUNNING'
    else:
        raise AssertionError('Overall completion timeout was ignored')
    dac.abort_ramp()
with connect_dac() as dac:
    assert dac.get_status().state=='ABORTED'
    held=dac.get_code(CHANNEL)
    # An incomplete line cannot produce a reply; transport timeout must close it.
    dac.write('ID',terminate=False)
    try:
        dac.read(timeout=0.05)
    except InstrumentTimeout:
        assert not dac.connected
    else:
        raise AssertionError('Missing reply did not raise a timeout')
    try: dac.get_id()
    except InstrumentError: pass
    else: raise AssertionError('Closed connection was silently reused')
with connect_dac() as dac:
    assert dac.get_id()=='AD5780'
    assert dac.get_code(CHANNEL)==held
    for malformed in ('SET 1 0\x00trailing','SET 1 0\nID','SET 1 0\rID'):
        try: dac.query(malformed)
        except ValueError: pass
        else: raise AssertionError('Embedded terminator/NUL accepted')
    assert dac.get_code(CHANNEL)==held
    # Unknown commands return multiline HELP: close rather than misalign replies.
    try: dac.query('UNKNOWN')
    except ProtocolError: assert not dac.connected
    else: raise AssertionError('Unexpected multiline reply was accepted')
with connect_dac() as fresh, Keysight34461A(address=METER_ADDRESS) as meter:
    assert fresh.get_id()=='AD5780'
    assert fresh.get_code(CHANNEL)==held
    meter.configure_dc_voltage(10,nplc=1)
    assert abs(meter.read_voltage()-fresh.get_voltage(CHANNEL))<VOLTAGE_TOL
    fresh.set_voltage(CHANNEL,0)
print('PASS P01: deadlines raise explicit errors, no automatic reconnect/replay, framing remains aligned')